# A2 · PyProj — putting shapes on the real Earth

**What PyProj is:** the Python door to **PROJ**, the library that knows every coordinate reference system (CRS)
and converts coordinates between them. PostGIS uses the same PROJ inside `ST_Transform`.

**Why you need it:** Shapely measures on flat paper. The Earth is round. A CRS is the recipe that flattens
the Earth onto paper. Choose the wrong recipe, and your distances and areas are wrong.

**Daily picture:** peeling an orange and pressing the peel flat. It always tears or stretches somewhere.
Each CRS chooses *where* it stretches:
- **Geographic CRS** (e.g. WGS 84, EPSG:4326): position in **degrees** (longitude, latitude), like GPS. Good for storing, bad for measuring.
- **Projected CRS** (e.g. UTM 33N, EPSG:32633): position in **metres** on a flat sheet. Good for measuring in a local area.

**Simple maths:** 1° of latitude ≈ 111 km everywhere. 1° of longitude ≈ 111 km × cos(latitude).
At Riverton (52.5° N), cos(52.5°) ≈ 0.61, so 1° of longitude is only ≈ 68 km. That is why "degrees" are not a length.

In [ ]:
import pyproj
from pyproj import CRS, Transformer, Geod
print("pyproj", pyproj.__version__, "| PROJ", pyproj.proj_version_str)

---
## Level 1 — Basics: what is a CRS?

**Goal of this level:** read a CRS and understand its name, type and units.

**Daily-life picture:** A postal address and a GPS reading describe the same house in two different 'languages'. A CRS is such a language.

### Exercise 1.1 — Look up a CRS by its EPSG code

**Step 1 · Purpose.** What is the **purpose** of `CRS.from_epsg(code)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

EPSG codes are ID numbers in a big catalogue (like ISBNs for books). 4326 = WGS 84 (GPS).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
wgs84 = CRS.from_epsg(____)
print(wgs84.name)
print(wgs84.axis_info)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Creates a CRS object from its EPSG number, so you can read its name, units and area of use.

**Real life:** Checking what 'EPSG:25832' means before using a dataset from a German city office.

```python
wgs84 = CRS.from_epsg(4326)
print(wgs84.name)
print(wgs84.axis_info)
```

Look at the axis order: officially **latitude first**, then longitude. Remember this for exercise 2.2.

</details>

### Exercise 1.2 — Geographic or projected?

**Step 1 · Purpose.** What is the **purpose** of `crs.is_geographic / crs.is_projected`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Compare 4326 with 32633 (UTM zone 33 North, the CRS of Riverton). Units: `crs.axis_info[0].unit_name`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
utm = CRS.from_epsg(____)
for c in [wgs84, utm]:
    print(c.name, "| geographic:", c.____, "| projected:", c.____, "| unit:", c.axis_info[0].unit_name)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Tells you whether the CRS uses degrees on the globe (geographic) or metres on a flat map (projected).

**Real life:** Before calculating a buffer of 500, check the data is projected; otherwise '500' means 500 degrees!

```python
utm = CRS.from_epsg(32633)
for c in [wgs84, utm]:
    print(c.name, "| geographic:", c.is_geographic, "| projected:", c.is_projected, "| unit:", c.axis_info[0].unit_name)
```

</details>

### Exercise 1.3 — Where may I use this CRS?

**Step 1 · Purpose.** What is the **purpose** of `crs.area_of_use`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Just read the attribute: `utm.area_of_use`. It has `.name` and `.bounds` (west, south, east, north).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
print(utm.____.name)
print(utm.area_of_use.____)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Shows the region of the world for which a CRS was designed (outside it, errors grow).

**Real life:** UTM 33N is made for 12°–18° E. Using it for Spain would stretch distances noticeably.

```python
print(utm.area_of_use.name)
print(utm.area_of_use.bounds)
```

</details>

---
## Level 2 — Core tools: transform coordinates

**Goal of this level:** convert a GPS position into metres and back, without the classic axis-order mistake.

**Daily-life picture:** Translating a sentence from one language to another and back: you should get the same sentence.

### Exercise 2.1 — Transform GPS degrees into metres

**Step 1 · Purpose.** What is the **purpose** of `Transformer.from_crs(a, b, always_xy=True).transform(x, y)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

From 4326 to 32633. With `always_xy=True` you always give **(lon, lat)** = (x, y). Riverton's town hall is at lon 13.4, lat 52.52.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
to_utm = Transformer.from_crs(____, ____, always_xy=True)
x, y = to_utm.transform(13.40, 52.52)
print(round(x), round(y))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Builds a converter between two CRSs and converts coordinates with it.

**Real life:** Your phone gives lon/lat of a bike accident; the city's map uses metres (UTM). You convert before comparing.

```python
to_utm = Transformer.from_crs(4326, 32633, always_xy=True)
x, y = to_utm.transform(13.40, 52.52)
print(round(x), round(y))
```

</details>

### Exercise 2.2 — The axis-order trap

**Step 1 · Purpose.** What is the **purpose** of `always_xy=False (the default)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Create a transformer **without** `always_xy`, and pass (13.40, 52.52) again. The result is far away because it reads 13.40 as latitude. Then pass (52.52, 13.40) instead.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
strict = Transformer.from_crs(4326, 32633)          # no always_xy
print(strict.transform(13.40, 52.52))                # wrong order for this transformer
print(strict.transform(____, ____))                  # correct order: lat, lon

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Shows that without `always_xy=True`, PyProj follows the official axis order of EPSG:4326: **latitude first**.

**Real life:** The most common bug in GIS: points land in the Indian Ocean or Somalia because lon and lat were swapped.

```python
strict = Transformer.from_crs(4326, 32633)
print(strict.transform(13.40, 52.52))
print(strict.transform(52.52, 13.40))
```

**Recommendation:** always write `always_xy=True`. Then the order is always x=lon, y=lat, like Shapely and GeoPandas.

</details>

### Exercise 2.3 — Back again (inverse)

**Step 1 · Purpose.** What is the **purpose** of `Transformer.from_crs(32633, 4326, always_xy=True)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Swap the two codes. Put the x, y from 2.1 in. You should get ≈ (13.40, 52.52).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
to_gps = Transformer.from_crs(____, ____, always_xy=True)
print(to_gps.transform(x, y))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Converts metres back into lon/lat; a round trip should return the start point.

**Real life:** Sending your result (in metres) to a web map or app that expects GPS coordinates.

```python
to_gps = Transformer.from_crs(32633, 4326, always_xy=True)
print(to_gps.transform(x, y))
```

</details>

### Exercise 2.4 — Find the right UTM zone

**Step 1 · Purpose.** What is the **purpose** of `pyproj.database.query_utm_crs_info()`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Simple maths: UTM zone = floor((lon + 180) / 6) + 1. For lon 13.4: (193.4 / 6) = 32.2 → zone 33. Use `AreaOfInterest(west, south, east, north)` with the same lon/lat four times.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
import math
from pyproj.aoi import AreaOfInterest
from pyproj.database import query_utm_crs_info
lon, lat = 13.40, 52.52
print("zone by formula:", math.floor((lon + 180) / 6) + 1)
info = query_utm_crs_info(datum_name="WGS 84", area_of_interest=AreaOfInterest(lon, lat, ____, ____))
print(info[0].code, info[0].name)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Asks the PROJ database which UTM CRS fits a given location.

**Real life:** You receive GPS data from a new city and need a metre-based CRS for it.

```python
import math
from pyproj.aoi import AreaOfInterest
from pyproj.database import query_utm_crs_info
lon, lat = 13.40, 52.52
print("zone by formula:", math.floor((lon + 180) / 6) + 1)
info = query_utm_crs_info(datum_name="WGS 84", area_of_interest=AreaOfInterest(lon, lat, lon, lat))
print(info[0].code, info[0].name)
```

GeoPandas has a shortcut: `gdf.estimate_utm_crs()` (notebook A3).

</details>

---
## Level 3 — Combining: measure on the globe and reproject shapes

**Goal of this level:** measure true distances and areas, and move whole Shapely shapes between CRSs.

**Daily-life picture:** Measuring a flight Berlin → New York with a string on a globe, not with a ruler on a flat map.

### Exercise 3.1 — True distance on the ellipsoid

**Step 1 · Purpose.** What is the **purpose** of `Geod(ellps='WGS84').inv(lon1, lat1, lon2, lat2)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`inv` returns (forward azimuth, back azimuth, distance in metres). Berlin (13.40, 52.52) → Paris (2.35, 48.86) is about 880 km.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
geod = Geod(ellps="WGS84")
az12, az21, dist = geod.____(13.40, 52.52, 2.35, 48.86)
print(round(dist / 1000), "km, heading", round(az12), "degrees")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Computes the shortest distance (and directions) between two lon/lat points on the curved Earth.

**Real life:** Flight or shipping distances, or checking long distances that a local UTM map would distort.

```python
geod = Geod(ellps="WGS84")
az12, az21, dist = geod.inv(13.40, 52.52, 2.35, 48.86)
print(round(dist / 1000), "km, heading", round(az12), "degrees")
```

</details>

### Exercise 3.2 — Reproject a whole Shapely shape

**Step 1 · Purpose.** What is the **purpose** of `shapely.ops.transform(transformer.transform, geom)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Pass the *function* `to_utm.transform` (no brackets!) and the polygon. Then `.area` is in m².

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
from shapely import Polygon
from shapely.ops import transform
area_ll = Polygon([(13.39, 52.51), (13.41, 52.51), (13.41, 52.53), (13.39, 52.53)])
area_m = transform(____, area_ll)
print("in degrees²:", area_ll.area, "| in m²:", round(area_m.area))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Applies a coordinate conversion to every vertex of a Shapely geometry.

**Real life:** You drew a study area in Google Maps (lon/lat) and need it in metres to compute its area.

```python
from shapely import Polygon
from shapely.ops import transform
area_ll = Polygon([(13.39, 52.51), (13.41, 52.51), (13.41, 52.53), (13.39, 52.53)])
area_m = transform(to_utm.transform, area_ll)
print("in degrees²:", area_ll.area, "| in m²:", round(area_m.area))
```

</details>

### Exercise 3.3 — Area directly on the ellipsoid

**Step 1 · Purpose.** What is the **purpose** of `geod.geometry_area_perimeter(geom)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Use `area_ll` from 3.2. The area comes out **negative** if the ring runs clockwise — take `abs()`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
a, p = geod.____(area_ll)
print(round(abs(a)), "m² on the ellipsoid vs", round(area_m.area), "m² in UTM")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Computes the true area and perimeter of a lon/lat Shapely geometry on the curved Earth.

**Real life:** Areas of big regions (countries, forests) where any flat map would distort the result.

```python
a, p = geod.geometry_area_perimeter(area_ll)
print(round(abs(a)), "m² on the ellipsoid vs", round(area_m.area), "m² in UTM")
```

Very close: UTM is excellent for local work. The difference grows for big areas or far from the zone centre.

</details>

### Exercise 3.4 — Convert a bounding box

**Step 1 · Purpose.** What is the **purpose** of `transformer.transform_bounds(...)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Riverton's box in metres is (390000, 5818000, 396000, 5824000). Use `to_gps.transform_bounds(*box)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
box_m = (390_000, 5_818_000, 396_000, 5_824_000)
print(to_gps.____(*box_m))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Converts a whole box (minx, miny, maxx, maxy), sampling its edges so curved edges are covered.

**Real life:** Asking an online service for data 'inside the town' when the service wants the box in lon/lat.

```python
box_m = (390_000, 5_818_000, 396_000, 5_824_000)
print(to_gps.transform_bounds(*box_m))
```

</details>

---
## Level 4 — Professional: choosing the right CRS

**Goal of this level:** pick a CRS for the question, and prove how big the error of a bad choice is.

**Daily-life picture:** Choosing the right map for a trip: a city map for walking, a road atlas for driving, a globe for flights.

### Exercise 4.1 — How wrong is Web Mercator?

**Scenario.** Web maps (Google, OSM) use **Web Mercator, EPSG:3857**. A colleague measured Riverton's town area in EPSG:3857. How wrong is the result compared to UTM?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Transform `area_ll` to 3857 and compare the area with UTM. Theory: Mercator stretches lengths by 1/cos(lat), so **areas** by 1/cos²(lat). At 52.5°: 1/0.61² ≈ 2.7 — almost 3 times too big!

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
import math
to_merc = Transformer.from_crs(4326, ____, always_xy=True)
area_merc = transform(to_merc.transform, area_ll)
print("ratio:", round(area_merc.area / area_m.area, 2), "| theory:", round(1 / math.cos(math.radians(52.52))**2, 2))

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Measurement (error analysis)

Measurement error analysis. Web Mercator is for **display**, never for measuring. That is why Greenland looks as big as Africa on web maps.

```python
import math
to_merc = Transformer.from_crs(4326, 3857, always_xy=True)
area_merc = transform(to_merc.transform, area_ll)
print("ratio:", round(area_merc.area / area_m.area, 2), "| theory:", round(1 / math.cos(math.radians(52.52))**2, 2))
```

</details>

### Exercise 4.2 — Which CRS for which question?

**Scenario.** Match each task with a CRS: (a) population density of all EU regions, (b) distances inside Riverton, (c) storing GPS tracks from an app,
(d) a background web map. Options: EPSG:3035 (LAEA Europe, *equal-area*), EPSG:32633 (UTM 33N), EPSG:4326 (WGS 84), EPSG:3857 (Web Mercator).
Then confirm with code that EPSG:3035 is equal-area.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Density = people ÷ **area**, so you need an equal-area CRS for a big region. Check a CRS's projection method with `CRS.from_epsg(3035).coordinate_operation.method_name`.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
print(CRS.from_epsg(____).coordinate_operation.method_name)

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Decision (method choice)

(a) 3035 — equal-area keeps areas right across Europe. (b) 32633 — local, metres, small distortion. (c) 4326 — universal storage format of GPS. (d) 3857 — what web tiles use, display only.

```python
print(CRS.from_epsg(3035).coordinate_operation.method_name)
```

</details>

### Exercise 4.3 — Joining GPS data to a local map

**Scenario.** A cycling app recorded a crash at lon 13.3955, lat 52.5310. The city's clinic *Central Clinic* is at x=392 900, y=5 821 100 (EPSG:32633). How far is the crash from the clinic, in metres?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Never mix CRSs. Transform the GPS point to 32633 first (`always_xy=True`), then use Pythagoras: √(dx² + dy²), like the diagonal of a room.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
cx, cy = to_utm.transform(____, ____)
print(round(math.hypot(cx - 392_900, cy - 5_821_100)), "m")

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Data integration (measurement)

Transform first, then measure. Mixing degrees and metres in one calculation is the second most common GIS bug (after swapped lon/lat).

```python
cx, cy = to_utm.transform(13.3955, 52.5310)
print(round(math.hypot(cx - 392_900, cy - 5_821_100)), "m")
```

</details>

---
## 🏁 Final test

Answer in this order: **question type → plan in words → code → interpretation.**

### Test 1 — 🛠️ Task question

**A.** A partner sends you Riverton's school locations as GPS: Oak Primary at (13.42723, 52.52504). Convert it to UTM 33N and check it lies inside the town box (390 000–396 000, 5 818 000–5 824 000).

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
x, y = Transformer.from_crs(4326, 32633, always_xy=True).transform(13.42723, 52.52504)
print(round(x), round(y), 390_000 <= x <= 396_000 and 5_818_000 <= y <= 5_824_000)
```

</details>

### Test 2 — 🛠️ Task question

**B.** Riverton wants to twin with a town at lon 21.01, lat 52.23 (Warsaw area). How far is it from Riverton town hall (13.40, 52.52) along the Earth's surface?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

Type: measurement on the globe (the two towns are in different UTM zones, so use the ellipsoid).
```python
print(round(Geod(ellps="WGS84").inv(13.40, 52.52, 21.01, 52.23)[2] / 1000), "km")
```

</details>

### Test 3 — 🛠️ Task question

**C.** A dataset arrives with coordinates like (5 820 000, 391 000). What is probably wrong and how do you check it?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

The numbers look like UTM but **x and y are swapped** (northing first). Check by plotting, or by transforming both orders to lon/lat and seeing which one lands where you expect.
Fix: swap the columns before building points, or (for a whole GeoDataFrame) use `shapely.ops.transform(lambda x, y: (y, x), geom)`.

</details>

### Test 4 — 🗺️ Decision question

**D.** You compare the green area per inhabitant in 30 European cities. Which CRS (or CRSs) do you use and why?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

Either one equal-area CRS for all (EPSG:3035 LAEA Europe) so every m² is comparable, or each city in its own UTM zone.
**Recommendation:** EPSG:3035, because one CRS for all cities makes the comparison simple and fair, and it is designed to keep areas right.

</details>

### Test 5 — 🧠 Modelling question

**E · Modelling.** You model the 'service area' of a regional hospital that covers a 60 km radius, crossing two UTM zones. How do you make sure the circle is a true 60 km everywhere?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

A buffer of 60 000 in one UTM zone is slightly distorted in the neighbouring zone. Options:
1. Use a **custom local projection** centred on the hospital, e.g. Azimuthal Equidistant: `CRS.from_proj4("+proj=aeqd +lat_0=52.5 +lon_0=13.4 +units=m")`. Distances **from the centre** are true in all directions. Buffer there, then transform back.
2. Or compute points on the ellipsoid with `Geod.fwd` every few degrees of heading.
Recommendation: option 1 — it is short, exact for 'distance from one point', and works with Shapely/GeoPandas.

</details>

---
## 🏗️ Project — Cyclists' GPS tracks: from degrees to real distances

**The brief.** Five volunteers recorded their morning bike rides with a phone app (one point every 10 s, in WGS84 lon/lat).
        The city wants each ride's length, duration and speed, a check of how wrong the numbers would be in the wrong CRS,
        and the tracks on a map. Use **PyProj** for all coordinate work. **Database note:** the last task publishes your result to PostGIS with `publish()` (a small helper around `to_postgis`, explained in B4). The database is the one you started in `00_START_HERE`. If it is not running yet, skip that code and open the saved GeoPackage in QGIS instead.

**Your data** (made for this project, in `data/projects/`):

| Layer / file | What it holds |
|---|---|
| `gps_tracks.csv` | columns `track_id`, `time`, `lon`, `lat` (EPSG:4326) |

**How to work:** for each task write the plan in words, then the code, then one sentence with the result. Model answers are hidden; open them only after you tried. The last part happens in **QGIS**.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

from geotrain.projects import build_projects, PROJ_DIR, publish
build_projects()                      # writes data/projects/ the first time
print(sorted(p.name for p in PROJ_DIR.iterdir()))

import numpy as np, pandas as pd, math
gps = pd.read_csv(PROJ_DIR / "gps_tracks.csv", parse_dates=["time"])
gps.head()

### Project task 1

**First look.** Number of points, start time and duration (minutes) per track.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
summary = gps.groupby("track_id").agg(points=("lon", "size"), start=("time", "min"), end=("time", "max"))
summary["minutes"] = (summary["end"] - summary["start"]).dt.total_seconds() / 60
summary
```

</details>

### Project task 2

**Length and speed in UTM.** Transform to EPSG:32633 (`always_xy=True`), sum the segment lengths with Pythagoras, and compute the mean speed in km/h.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
to_utm = Transformer.from_crs(4326, 32633, always_xy=True)
gps["x"], gps["y"] = to_utm.transform(gps["lon"].values, gps["lat"].values)
def length_m(g):
    return float(np.hypot(np.diff(g["x"]), np.diff(g["y"])).sum())
summary["km_utm"] = gps.groupby("track_id").apply(length_m, include_groups=False) / 1000
summary["kmh"] = summary["km_utm"] / (summary["minutes"] / 60)
summary[["km_utm", "minutes", "kmh"]].round(2)
```
GPS noise (a few metres per point) makes zig-zags, so tracks look slightly longer than the real route.

</details>

### Project task 3

**Check with the ellipsoid.** Compute each track's length with `Geod.line_length(lons, lats)` and compare with UTM (difference in metres).

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
geod = Geod(ellps="WGS84")
summary["km_geod"] = gps.groupby("track_id").apply(lambda g: geod.line_length(g["lon"], g["lat"]), include_groups=False) / 1000
summary["diff_m"] = (summary["km_utm"] - summary["km_geod"]) * 1000
summary[["km_utm", "km_geod", "diff_m"]].round(3)
```
A few metres on several km: UTM is fine locally.

</details>

### Project task 4

**The wrong CRS.** Compute the lengths in Web Mercator (EPSG:3857). By what factor are they too long? Compare with the theory 1/cos(latitude).

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
to_merc = Transformer.from_crs(4326, 3857, always_xy=True)
mx, my = to_merc.transform(gps["lon"].values, gps["lat"].values)
gps["mx"], gps["my"] = mx, my
km_merc = gps.groupby("track_id").apply(lambda g: np.hypot(np.diff(g["mx"]), np.diff(g["my"])).sum(), include_groups=False) / 1000
print((km_merc / summary["km_utm"]).round(3).tolist(), "theory:", round(1 / math.cos(math.radians(gps["lat"].mean())), 3))
```

</details>

### Project task 5

**Tracks as lines, which cross the river?** Build one LineString per track (UTM), test `intersects` with the river, and publish lines and points.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
from shapely import LineString as LS
import geopandas as gpd
river_line = gpd.read_file(GPKG, layer="river").geometry.iloc[0]
tracks = gpd.GeoDataFrame([{"track_id": t, "km": round(summary.loc[t, "km_utm"], 2), "kmh": round(summary.loc[t, "kmh"], 1),
                            "geometry": LS(list(zip(g["x"], g["y"])))} for t, g in gps.groupby("track_id")], crs=32633)
tracks["crosses_river"] = tracks.intersects(river_line)
points = gpd.GeoDataFrame(gps[["track_id", "time"]], geometry=gpd.points_from_xy(gps["x"], gps["y"]), crs=32633)
tracks.to_file(PROJ_DIR / "a2_tracks.gpkg", layer="tracks"); points.to_file(PROJ_DIR / "a2_tracks.gpkg", layer="points")
publish({"gps_tracks": tracks, "gps_points": points})
tracks.drop(columns="geometry")
```

</details>

### 🗺️ Project in QGIS

1. Load `gps_tracks` and `gps_points` (PostGIS schema `projects`, or `a2_tracks.gpkg`).
2. Style `gps_tracks` by `track_id` (*Categorized*), line width 1 mm; label with `kmh`.
3. **Animate the rides:** open `gps_points` → *Layer Properties → Temporal* → tick *Dynamic Temporal Control*,
   configuration *Single Field with Date/Time* = `time`. Then *View → Panels → Temporal Controller* → press ▶.
4. Set the project CRS to EPSG:3857 and use the *Measure Line* tool on a track: QGIS measures on the ellipsoid by default — compare with your numbers.

<details><summary>🔌 <b>How to connect QGIS to PostGIS</b> (once per computer)</summary>

1. Start the database (Docker: `docker start geotrain-db`).
2. In QGIS, open the **Browser** panel → right-click **PostgreSQL** → **New Connection…**
3. Name `geotrain`, Host `localhost`, Port `5432`, Database `geotrain`.
   Authentication → *Basic*: user `geo`, password `geo` (tick *Store*). Click **Test Connection** → OK.
4. Expand **geotrain → projects** (the schema) and **drag a table onto the map**.
5. To run SQL and see the result on the map: **Database → DB Manager → PostGIS → geotrain → SQL Window**.
   Write the query → **Execute** → tick **Load as new layer**, choose the geometry column and a unique id column → **Load**.
6. To open a GeoPackage instead: drag the `.gpkg` file from your file manager into QGIS.

</details>

### 📦 Deliverables (tick when done)

- [ ] Table per track: points, minutes, km (UTM and ellipsoid), km/h, crosses river
- [ ] Explanation (2 sentences) of the Web Mercator error
- [ ] QGIS animation (screenshot) of the rides

### 🪞 Reflection — your long-term plan

1. Which CRS will you use for your own study area? Look it up with `query_utm_crs_info` or your national mapping agency.
2. Write down one dataset you already have and its CRS. Is it geographic or projected?

✍️ **Your answer:** …